# 04b · Dipéptido de alanina: protocolo de cierre

Mismo protocolo que el Swiss roll (05), el toro (02b) y MNIST (03b): DMAP congelado sobre distancias entre átomos pesados, `vae` con kl = 10⁻³ y 10 000 pasos, desde cero y por continuación en m, recuperación de la carta (φ, ψ) con kNN frente a su piso, tasas con Clopper-Pearson al 95 %.

**Réplicas (cambio de v5).** Las particiones siguen siendo trayectorias completas (sin fuga temporal). Con `trajectory_rotation`, cada semilla elige qué trayectoria es entrenamiento / validación / test (las 6 permutaciones) y la fase del submuestreo regular; 12 semillas recorren cada permutación dos veces con fases distintas.

**Operador.** α = 1 (Laplace-Beltrami, el mismo que en las demás geometrías). El `config_alanine.json` antiguo usaba α = 0.5; si el texto del artículo usa 0.5, cambia `ALPHA` y decláralo.

Hipótesis y criterios, **fijados antes de correr**:

| | Pregunta | Diseño | Criterio |
|---|---|---|---|
| **L1** | ¿En qué m el prefijo contiene la carta (φ, ψ)? | oráculo afín, 12 réplicas | se define m_info = menor m con mediana del oráculo ≤ 2 × su valor en m = M. Sin predicción cerrada: la densidad no es uniforme y el espacio no es un producto |
| **L2** | ¿La continuación espectral despliega la carta? | latente 2 (12 réplicas) y latente 4 (6 réplicas) | `vae_warm`: límite inferior Clopper-Pearson > 0.6 para todo m ≥ m_info |
| **L3** | ¿Ayuda la continuación frente a entrenar desde cero? | mismas réplicas | Fisher exacto unilateral en m = M |
| **L4** | Comparador convencional | `plainvae` (MLP sobre las coordenadas), latentes 2 y 4 | misma métrica |
| **L5** | Generación condicionada (supervisada, aparte) | `cvae` en m = M, 2 réplicas | error angular pedido vs medido; candidatos no validados físicamente |

La carta de Ramachandran es topológicamente un toro, pero la densidad se concentra en cuencas: un latente 2 puede desplegar las regiones pobladas sin que exista una carta global. El orden aleatorio ya se probó en el Swiss roll y el toro; aquí no se repite.

β sin etiquetas en dos etapas (lectura afín de x y confirmación con `direct`, ambas en validación). Duración estimada ~4 h en dos T4.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo, métrica kNN, continuación con ceros, paso AE→VAE y los controles nuevos de v5 (regla de degeneración del toro, regla argmin, sonda de etiquetas de MNIST, rotación de trayectorias de alanina). Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_04b.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_04b.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## Datos y β sin etiquetas (mismo n que el experimento, ~25 min)

Datos: las tres trayectorias de 250 ns de mdshare (FU Berlin, 1 ps por fotograma, 10 átomos pesados, nm). Si no hay un NPZ preparado en `/kaggle/input`, la celda lo construye con `prepare_alanine.py --mdshare`. Usa los dos NPZ de mdshare si los subiste como Dataset y, si no, los descarga (hace falta **Internet: On**). Los índices de φ y ψ no se suponen: se identifican como las únicas cuádruplas enlazadas que reproducen el archivo publicado de diedros en todos los fotogramas. Las columnas `oracle_*` usan (φ, ψ) y son solo diagnóstico; la masa de atajos del grafo se reporta, no se usa.

In [ ]:
import time
import numpy as np
import pandas as pd
from IPython.display import display, Image
import protocol as P

N = 3000                       # 1000 fotogramas por trayectoria: n_train = 1000 > batch = 512
ALPHA = 1.0
# 1) un NPZ ya preparado (xyz, trajectory, phi/psi, bonds) en /kaggle/input o de una sesión anterior;
# 2) si no, se construye desde mdshare (3 x 250 ns, átomos pesados): usa los dos NPZ de mdshare si
#    están en /kaggle/input y, si no, los descarga (Settings -> Internet: On).
def es_preparado(p):
    try:
        with np.load(p) as z:
            return {'xyz', 'trajectory', 'phi_indices', 'psi_indices', 'bonds'} <= set(z.files)
    except Exception:
        return False
found = [p for p in sorted(Path('/kaggle/input').rglob('*.npz')) if es_preparado(p)]
ALANINE = str(found[0]) if found else '/kaggle/working/alanine.npz'
if not es_preparado(ALANINE):
    t0 = time.time()
    r = subprocess.run([sys.executable, str(ROOT / 'prepare_alanine.py'), '--mdshare', '--out', ALANINE,
                        '--search', '/kaggle/input', '--mdshare-dir', '/kaggle/working/mdshare'],
                       capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode:
        print(r.stderr[-3000:])
        raise RuntimeError('No se pudo construir alanine.npz desde mdshare. Activa Internet en el notebook '
                           '(Settings -> Internet: On) o sube como Dataset los archivos '
                           'alanine-dipeptide-3x250ns-heavy-atom-positions.npz y '
                           'alanine-dipeptide-3x250ns-backbone-dihedrals.npz.')
    print(f'alanine.npz construido en {(time.time()-t0)/60:.1f} min')
a = np.load(ALANINE)
traj, counts = np.unique(a['trajectory'], return_counts=True)
print(f'{ALANINE}: {a["xyz"].shape[1]} átomos, trayectorias {dict(zip(traj.tolist(), counts.tolist()))}')
assert len(traj) >= 3 and counts[:3].min() >= N // 3, 'Se necesitan 3 trayectorias con al menos N/3 fotogramas.'

cfg_sweep = json.loads((ROOT / 'config_beta_sweep_alanine.json').read_text())
cfg_sweep.update(n=N, data=ALANINE, alpha=ALPHA)
sel = P.run_sweep(cfg_sweep, ROOT, 'beta_sweep_alanine')
assert sel['selected_factor'] is not None, 'Ningún factor pasó los controles espectrales: amplía la lista.'
SWEEP = Path(cfg_sweep['out'])
sw = pd.read_csv(SWEEP / 'beta_sweep.csv')
SEL_MODES = cfg_sweep['selection_modes']
cols = [c for c in ['kernel_length', 'ok', 'graph_shortcut_mass', 'perron_gap', 'min_eigenvalue',
                    'nystrom_training_relative_error', 'ambient_val_selection']
        + [f'ambient_val_m{m}' for m in SEL_MODES] + [f'oracle_val_m{m}' for m in SEL_MODES] if c in sw]
display(sw.groupby('factor')[cols].mean().style.format('{:.3g}').set_caption(
    f"etapa 1: seleccionado {sel['selected_factor']}, banda {sel['equivalent_band']}"))
display(Image(filename=str(SWEEP / 'beta_sweep.png')))

# ---- etapa 2: confirmación con el decoder direct (validación de x, sin la carta) ----
BASE = json.loads((ROOT / 'config_alanine.json').read_text())
BASE.update(n=N, alpha=ALPHA, data=ALANINE, trajectory_rotation=True)
COMMON = dict(P.COMMON)
if sel.get('edge_warning'):
    print('ATENCIÓN (etapa 1):', sel['edge_warning'])
# La lectura afín de la etapa 1 favorece núcleos anchos (autofunciones casi lineales), así que la
# etapa 2 NO se limita a sus vecinos: 4 candidatos repartidos en toda la rejilla admisible.
CAND = P.spread(sw, k=4)
CONF = [dict(tag=f'beta{f:g}', beta_factor=f, seeds=[0, 1], modes=SEL_MODES, models=['direct'],
             m_protocols=['scratch'], oracle_decoder=False, out=f'/kaggle/working/beta_confirm_alanine/f{f:g}')
        for f in CAND]
print('candidatos etapa 2 (repartidos en la rejilla):', CAND)
P.run_plan(CONF, BASE, ROOT, common=COMMON)
conf, FACTOR = P.confirm_direct({p['beta_factor']: p['out'] for p in CONF}, SEL_MODES)
display(conf.style.format('{:.4g}').set_caption('etapa 2: MSE de validación del decoder direct (media sobre semillas y m)'))
print(f"factor final = {FACTOR} (etapa 1: {sel['selected_factor']})")

## Plan y estimación de tiempo

Con n_train = 1000 y batch 512 hay 2 pasos por época, así que 10 000 pasos son 5 000 épocas con validación en cada una: el costo por modelo es algo mayor que en el Swiss roll. Ajusta `SECONDS_PER_MODEL` con el tiempo real.

In [ ]:
import math
import protocol as P
COMMON = dict(P.COMMON)
LADDER = [2, 4, 8, 16, 32, 64]
M = max(LADDER)
PLAN = [
    dict(tag='A_k2_spectral', latent=2, beta_factor=FACTOR, seeds=list(range(12)), modes=LADDER,
         models=['vae'], m_protocols=['scratch', 'warm'], chart_decoder_epochs=10_000),
    dict(tag='B_k4_spectral', latent=4, beta_factor=FACTOR, seeds=list(range(6)), modes=LADDER,
         models=['vae'], m_protocols=['scratch', 'warm']),
    dict(tag='C_plainvae_k2', latent=2, beta_factor=FACTOR, seeds=list(range(6)), modes=[LADDER[0]],
         models=['plainvae'], m_protocols=['scratch'], oracle_decoder=False),
    dict(tag='C_plainvae_k4', latent=4, beta_factor=FACTOR, seeds=list(range(6)), modes=[LADDER[0]],
         models=['plainvae'], m_protocols=['scratch'], oracle_decoder=False),
    dict(tag='D_cvae_k2', latent=2, beta_factor=FACTOR, seeds=list(range(2)), modes=[M],
         models=['cvae'], m_protocols=['scratch'], oracle_decoder=False),
]
SECONDS_PER_MODEL = 100
print('pasos de Adam por modelo =', max(COMMON['epochs'] * math.ceil((N // 3) / COMMON['batch']), COMMON['min_steps']))
P.show_plan(PLAN, 'alanine', SECONDS_PER_MODEL, COMMON)

## Ejecutar

Se puede re-ejecutar: las configuraciones completas se omiten y las parciales se señalan (bórralas o cambia el `tag`). Para una corrida larga conviene **Save Version → Save & Run All (Commit)**.

In [ ]:
import protocol as P
hechas, problemas = P.run_plan(PLAN, BASE, ROOT, common=COMMON)

## Análisis

"Desplegada" = kNN(latente → (cos φ, sin φ, cos ψ, sin ψ)) ≤ 10 × el piso del mismo estimador sobre la carta verdadera del test. Las curvas incluyen el inicio de cada cadena `warm` (= modelo desde cero en m = 2).

In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from IPython.display import display, Image
import protocol as P
R = {p['tag']: P.load(p['out']) for p in PLAN}
A, B = R['A_k2_spectral'], R['B_k4_spectral']
fmt = {'tasa': '{:.0%}', 'ic95_inf': '{:.2f}', 'ic95_sup': '{:.2f}', 'mse_x_mediana': '{:.4g}',
       'knn_ratio_mediana': '{:.3g}'}

# ---- L1: información del prefijo ----
M_INFO = None
if A is not None:
    info = P.information_table(A).join(P.information_table(A, 'mse_x', 'affine_x').add_prefix('ambiente_'))
    display(info.style.format('{:.3g}').set_caption('L1 · oráculo afín (carta) y lectura afín de x, cuartiles'))
    med = info['mediana']
    M_INFO = int(min(m for m in med.index if med.loc[m] <= 2 * med.loc[M]))
    print(f'm_info (mediana del oráculo <= 2x su valor en m = {M}) = {M_INFO}')
    kf = A.knn_chart_floor.dropna()
    print(f'piso kNN de la carta: mediana {kf.median():.3g} (rango {kf.min():.3g}–{kf.max():.3g} entre réplicas)')

# ---- L2 / L4: despliegue ----
tablas = {}
for tag, lab in (('A_k2_spectral', 'A latente 2'), ('B_k4_spectral', 'B latente 4'),
                 ('C_plainvae_k2', 'plainvae latente 2'), ('C_plainvae_k4', 'plainvae latente 4')):
    if R[tag] is not None:
        tablas[tag] = P.unfold_table(R[tag], lab)
        display(tablas[tag].style.format(fmt).set_caption(lab))
for tag in ('A_k2_spectral', 'B_k4_spectral'):
    if tag in tablas and M_INFO is not None:
        t = tablas[tag]
        w = t[(t.model == 'vae_warm') & (t.m >= M_INFO)]
        print(f"Criterio L2 {tag} (IC inf > 0.6 para vae_warm, m >= {M_INFO}): "
              f"{'CUMPLIDO' if len(w) and (w.ic95_inf > .6).all() else 'NO cumplido'}")

# ---- L3: continuación frente a desde cero en m = M ----
for tag, d in (('A', A), ('B', B)):
    if d is None:
        continue
    q = P.trained(d)
    a_, b_ = q[(q.model == 'vae_warm') & (q.m == M)], q[(q.model == 'vae') & (q.m == M)]
    ka, kb = int(a_.unfolded.astype(bool).sum()), int(b_.unfolded.astype(bool).sum())
    p = fisher_exact([[ka, len(a_)-ka], [kb, len(b_)-kb]], alternative='greater')[1]
    pw = P.paired_warm_vs_scratch(d, M)
    print(f"L3 {tag} m={M}: warm {ka}/{len(a_)} vs desde cero {kb}/{len(b_)}, Fisher p = {p:.3g}"
          + (f" | MSE_x pareado: warm mejor en {pw['warm_mejor']}/{pw['n']}, Wilcoxon p = {pw['wilcoxon_p']:.3g}" if pw else ''))

# ---- error angular y diagnósticos moleculares de la reconstrucción en m = M ----
def med_list(s):
    v = np.array([ast.literal_eval(x) if isinstance(x, str) else x for x in s.dropna()])
    return np.median(v, 0) if len(v) else [np.nan, np.nan]
rows = []
for tag, d in R.items():
    if d is None:
        continue
    q = P.trained(d)
    for model in sorted(set(q.model)):
        g = q[(q.model == model) & (q.m == q[q.model == model].m.max())]
        e = med_list(g.angular_mae_deg) if 'angular_mae_deg' in g else [np.nan, np.nan]
        rows.append(dict(corrida=tag, model=model, m=int(g.m.iloc[0]), n=len(g), error_phi_grados=e[0], error_psi_grados=e[1],
                         enlaces_fuera=g.bond_outlier_fraction.median() if 'bond_outlier_fraction' in g else np.nan,
                         quiralidad=g.chirality_sign_error.median() if 'chirality_sign_error' in g else np.nan))
display(pd.DataFrame(rows).style.format('{:.3g}', subset=['error_phi_grados', 'error_psi_grados', 'enlaces_fuera', 'quiralidad'])
        .set_caption('Reconstrucción en la mayor m de cada corrida: error angular mediano y diagnósticos empíricos'))

In [ ]:
# ---- Figura resumen (2x2) ----
fig, ax = plt.subplots(2, 2, figsize=(12.5, 8.6))
(a, b), (c, e) = ax
if A is not None:
    o = A[A.model == 'oracle'].groupby('m').chart_mse_z
    a.plot(o.median().index, o.median().values, 'o-', color='C4', label='carta: oráculo afín (mediana)')
    a.fill_between(o.median().index, o.quantile(.25).values, o.quantile(.75).values, color='C4', alpha=.2, lw=0)
    x_ = A[A.model == 'affine_x'].groupby('m').mse_x.median()
    a.plot(x_.index, x_.values, 's--', color='gray', label='ambiente: lectura afín de x (mediana, nm²)')
    if M_INFO is not None:
        a.axvline(M_INFO, color='k', ls=':', lw=.8, label=f'm_info = {M_INFO}')
a.set(xscale='log', yscale='log', xlabel='m', ylabel='MSE', title='(a) Información del prefijo')
a.legend(fontsize=7)

estilos = [('A_k2_spectral', 'vae_warm', 'C4', '-', 'o', 'latente 2, continuación'),
           ('A_k2_spectral', 'vae', 'C0', ':', '^', 'latente 2, desde cero'),
           ('B_k4_spectral', 'vae_warm', 'C3', '-', 'D', 'latente 4, continuación'),
           ('B_k4_spectral', 'vae', 'C7', ':', 'v', 'latente 4, desde cero')]
for i, (tag, model, col, ls, mk, lab) in enumerate(estilos):
    if tag in tablas:
        P.errorbar_rates(b, tablas[tag], model, dx=1.03**(i-1.5), color=col, ls=ls, marker=mk, label=lab)
for tag, col, lab in (('C_plainvae_k2', 'C2', 'plainvae latente 2'), ('C_plainvae_k4', 'C8', 'plainvae latente 4')):
    if tag in tablas and len(tablas[tag]):
        b.axhline(tablas[tag].tasa.iloc[0], color=col, ls='-.', lw=1, label=f'{lab} ({tablas[tag].desplegadas.iloc[0]})')
if M_INFO is not None:
    b.axvline(M_INFO, color='k', ls=':', lw=.8)
b.set(xscale='log', ylim=(-.05, 1.05), xlabel='m', ylabel='fracción con carta desplegada',
      title='(b) Recuperación de (φ, ψ) (IC 95 %)')
b.legend(fontsize=7)

rng = np.random.default_rng(0)
for j, (tag, model, col, ls, mk, lab) in enumerate(estilos):
    if R.get(tag) is None:
        continue
    g = P.with_chain_start(R[tag])
    g = g[(g.model == model) & g.knn_chart_ratio.notna()]
    c.scatter(g.m*np.exp(rng.uniform(-.05, .05, len(g)) + (j-1.5)*.06), g.knn_chart_ratio, s=12, alpha=.6,
              color=col, marker=mk, label=lab)
c.axhline(float(COMMON['unfold_ratio']), color='k', ls=':', lw=.8, label='umbral (10× piso)')
c.set(xscale='log', yscale='log', xlabel='m', ylabel='kNN carta / piso', title='(c) Distribución (cada punto = réplica)')
c.legend(fontsize=7)

for tag, model, col, ls, mk, lab in estilos:
    if R.get(tag) is not None:
        q = P.with_chain_start(R[tag])
        q = q[q.model == model].groupby('m').mse_x.median()
        e.plot(q.index, q.values, color=col, ls=ls, marker=mk, label=lab)
for tag, col, lab in (('C_plainvae_k2', 'C2', 'plainvae latente 2'), ('C_plainvae_k4', 'C8', 'plainvae latente 4')):
    if R[tag] is not None:
        e.axhline(P.trained(R[tag]).mse_x.median(), color=col, ls='-.', lw=1, label=lab)
if A is not None and 'chart_decoder' in set(A.model):
    e.axhline(A[A.model == 'chart_decoder'].mse_x.median(), color='k', ls=':', lw=.8, label='decoder sobre (φ, ψ) verdaderos')
e.set(xscale='log', yscale='log', xlabel='m', ylabel='test MSE$_x$ (nm², mediana)', title='(d) Reconstrucción')
e.legend(fontsize=7)
for axis in ax.flat:
    axis.set_xticks(LADDER); axis.set_xticklabels([str(m) for m in LADDER], fontsize=8); axis.minorticks_off()
fig.tight_layout()
fig.savefig('/kaggle/working/alanine_protocolo.png', dpi=200); fig.savefig('/kaggle/working/alanine_protocolo.pdf')
plt.show()
if tablas:
    pd.concat(tablas.values()).to_csv('/kaggle/working/alanine_protocolo_tablas.csv', index=False)
for name in ('angles_oracle_reconstructed', 'angles_vae_reconstructed', 'angles_generated_vae_warm'):
    f = Path(PLAN[0]['out']) / f'{name}.png'
    if f.exists():
        print(name); display(Image(filename=str(f)))

## L5 · Generación condicionada (cvae, supervisada)

Las muestras son candidatos estructurales: se reporta el error entre los ángulos pedidos y los medidos. No se certifican energías ni validez física.

In [ ]:
ckpt = Path(PLAN[-1]['out']) / 'seed_0' / f'cvae_m{M}.pt'
if ckpt.exists():
    for phi, psi, region in ((-60, -40, 'alfa_R'), (-150, 150, 'beta'), (60, 40, 'alfa_L')):
        r = subprocess.run([sys.executable, str(ROOT / 'generate_angles.py'), str(ckpt), '--phi', str(phi), '--psi', str(psi),
                            '--n', '64', '--out', str(Path(PLAN[-1]['out']) / f'candidatos_{region}.npz')],
                           capture_output=True, text=True)
        print(region, r.stdout.strip() if not r.returncode else 'falló: ' + r.stderr[-500:])
else:
    print('No hay checkpoint cvae (D_cvae_k2 no se ejecutó).')

## Qué descargar

`/kaggle/working/results_alanine_*`, `beta_sweep_alanine/`, `beta_confirm_alanine/`, `alanine_protocolo.png/.pdf` y `alanine_protocolo_tablas.csv`. El `provenance.json` de cada semilla guarda qué trayectoria fue entrenamiento/validación/test y la fase del submuestreo.